# Tower of London

**Track:** Executive Functions
**Construct:** Multi-step planning and means-ends analysis

Tests multi-step planning ability by requiring the model to find optimal move sequences to rearrange balls on pegs to match a goal state.

## Cognitive Science Background

The **Tower of London** (Shallice, 1982) is a classic test of planning and problem-solving — a core executive function. It requires lookahead and means-ends analysis: mentally simulating move sequences to find the shortest path from an initial state to a goal state. Planning depth correlates with prefrontal cortex function (Owen et al., 1990) and is one of the most reliable markers of executive dysfunction.

**Human baseline:** ~85% optimal at 3 moves, ~55% at 5 moves (Owen et al., 1990).

## Methodology

Problems present an initial peg configuration and a goal configuration. The model must produce a move sequence that transforms the initial state into the goal state in the minimum number of moves. Difficulty tiers:


| Tier | Weight | Description |
|------|--------|-------------|
| Easy (2-move) | 0.20 | Minimal lookahead required |
| Medium (3-move) | 0.30 | Moderate planning depth |
| Hard (4–5-move) | 0.50 | Deep lookahead with subgoal management |


## Scoring

`Score = move_efficiency (optimal_moves / actual_moves)`

A score of 1.0 means the model always finds the optimal solution. Heavy weighting toward hard problems ensures discrimination.

| Score | Interpretation |
|:---:|---|
| 0.8–1.0 | Optimal planning — finds minimum-move solutions even at 4–5 move depth |
| 0.5–0.8 | Good — solves most problems but uses extra moves on hard items |
| 0.2–0.5 | Limited planning horizon — solves easy but inefficient on 3+ move problems |
| 0.0–0.2 | Minimal planning ability — cannot solve multi-step spatial rearrangements |

### References

Shallice (1982), Owen et al. (1990), Miyake et al. (2000)

In [ ]:
!pip install -q protobuf==5.29.6 kaggle-benchmarks numpy 2>/dev/null

In [ ]:
"""
Stimuli generator for Tower of London (ToL) planning benchmark.

Generates goal states at varying optimal move depths (2, 3, 4, 5 moves).
Uses 3 pegs and 3 colored balls. Each peg has a capacity constraint:
- Peg A: holds 3 balls
- Peg B: holds 2 balls
- Peg C: holds 1 ball

This matches the classic Shallice (1982) setup.
"""

import random
from collections import deque
from copy import deepcopy

# Pegs with capacity constraints
PEG_CAPACITY = {"A": 3, "B": 2, "C": 1}
BALLS = ["red", "blue", "green"]


def state_to_tuple(state):
    """Convert state dict to hashable tuple."""
    return tuple(tuple(state[p]) for p in ["A", "B", "C"])


def tuple_to_state(t):
    """Convert tuple back to state dict."""
    return {"A": list(t[0]), "B": list(t[1]), "C": list(t[2])}


def get_valid_moves(state):
    """Get all valid moves from current state."""
    moves = []
    pegs = ["A", "B", "C"]
    for src in pegs:
        if not state[src]:
            continue
        ball = state[src][-1]
        for dst in pegs:
            if dst == src:
                continue
            if len(state[dst]) < PEG_CAPACITY[dst]:
                moves.append((src, dst, ball))
    return moves


def apply_move(state, move):
    """Apply a move and return new state."""
    src, dst, ball = move
    new_state = deepcopy(state)
    new_state[src].pop()
    new_state[dst].append(ball)
    return new_state


def bfs_optimal(start, goal):
    """Find optimal (shortest) move sequence from start to goal using BFS."""
    start_t = state_to_tuple(start)
    goal_t = state_to_tuple(goal)

    if start_t == goal_t:
        return []

    queue = deque([(start_t, [])])
    visited = {start_t}

    while queue:
        current_t, path = queue.popleft()
        current = tuple_to_state(current_t)

        for move in get_valid_moves(current):
            new_state = apply_move(current, move)
            new_t = state_to_tuple(new_state)

            new_path = path + [move]
            if new_t == goal_t:
                return new_path

            if new_t not in visited:
                visited.add(new_t)
                queue.append((new_t, new_path))

    return None


def generate_all_states():
    """Generate all valid states (3 balls distributed across 3 pegs respecting capacity)."""
    states = []
    pegs = ["A", "B", "C"]

    def place_balls(balls_remaining, current_state):
        if not balls_remaining:
            states.append(deepcopy(current_state))
            return
        ball = balls_remaining[0]
        for peg in pegs:
            if len(current_state[peg]) < PEG_CAPACITY[peg]:
                current_state[peg].append(ball)
                place_balls(balls_remaining[1:], current_state)
                current_state[peg].pop()

    place_balls(BALLS, {"A": [], "B": [], "C": []})
    return states


def generate_tol_problems(n_per_depth=5, seed=42):
    """
    Generate Tower of London problems at depths 2, 3, 4, and 5.
    Returns problems grouped by optimal move count.
    """
    random.seed(seed)
    all_states = generate_all_states()

    problems_by_depth = {2: [], 3: [], 4: [], 5: []}

    for start in all_states:
        for goal in all_states:
            if state_to_tuple(start) == state_to_tuple(goal):
                continue
            optimal = bfs_optimal(start, goal)
            if optimal and len(optimal) in problems_by_depth:
                problems_by_depth[len(optimal)].append({
                    "start": deepcopy(start),
                    "goal": deepcopy(goal),
                    "optimal_moves": len(optimal),
                    "optimal_solution": [(s, d, b) for s, d, b in optimal],
                })

    problems = []
    for depth in [2, 3, 4, 5]:
        candidates = problems_by_depth[depth]
        random.shuffle(candidates)
        selected = candidates[:n_per_depth]
        for i, p in enumerate(selected):
            p["problem_id"] = f"tol_{depth}move_{i+1}"
        problems.extend(selected)

    return problems


def state_str(state):
    """Human-readable state description."""
    parts = []
    for peg in ["A", "B", "C"]:
        if state[peg]:
            balls = ", ".join(state[peg])
            parts.append(f"Peg {peg}: [{balls}] (bottom→top)")
        else:
            parts.append(f"Peg {peg}: [empty]")
    return "\n".join(parts)


TOL_PROBLEMS = generate_tol_problems(n_per_depth=5, seed=42)

In [ ]:
"""
Executive Functions Benchmark 2: Tower of London (ToL) Planning

Tests planning ability — a core executive function component.

The model is given an initial arrangement of 3 colored balls on 3 pegs (with
capacity constraints) and a goal state. It must plan a sequence of moves to
reach the goal in the minimum number of moves.

Cognitive Science Basis:
- Tower of London (Shallice, 1982)
- Planning is a "look-ahead" executive process (Owen et al., 1990)
- Difficulty scales with optimal move depth (2 < 3 < 4 < 5 moves)
- Frontal patients show deficits at higher move depths (Shallice, 1982)

Metrics:
- Per-tier mean optimality (optimal_moves / actual_moves if goal reached, else 0)
- Three tiers: Easy (2-move, 0.20), Medium (3-move, 0.30), Hard (4-5 move, 0.50)
- Score = weighted sum of per-tier mean optimality

Shortcut Resistance:
- Problems are procedurally generated, not from standard test batteries
- Capacity constraints prevent trivial solutions
- Multiple move depths test genuine planning vs. random search
"""

import kaggle_benchmarks as kbench
import json as _json
def _safe_log(data): print(_json.dumps(data, indent=2, default=str))
import numpy as np
import re
from copy import deepcopy


# ─── Move Parsing ───────────────────────────────────────────────────


def _strip_think(text: str) -> str:
    """Remove <think>...</think> blocks from model output."""
    return re.sub(r'<think>.*?</think>', '', text, flags=re.DOTALL).strip()

def parse_moves(text) -> list:
    """Parse move list from model response into (src, dst) tuples.
    
    Strategy (ordered by reliability):
    1. Find a MOVES: summary line and parse only that line
    2. Find numbered move lines (Move 1: A→B) and extract one move per line
    3. Find the last compact move list on a single line (A→B, B→C)
    Never fall back to full-text search — that picks up reasoning traces.
    """
    if not isinstance(text, str):
        text = str(text)
    
    _arrow = r'(?:→|->|—>|=>)'
    _move_pat = rf'\b([ABC])\s*{_arrow}\s*([ABC])\b'
    
    # === Strategy 1: MOVES: summary line ===
    moves_match = re.search(r'MOVES:\s*(.+)', text, re.IGNORECASE)
    if moves_match:
        line = moves_match.group(1)
        direct = re.findall(_move_pat, line, re.IGNORECASE)
        if direct:
            return [(s.upper(), d.upper()) for s, d in direct]
    
    # === Strategy 2: Numbered move lines ===
    # Match patterns like "Move 1: A→B", "**Move 1:** A→B", "1. A→B", "Step 1: A→B"
    numbered = re.findall(
        rf'(?:(?:Move|Step)\s*\d+[:\.]?\s*\**\s*|\d+\.\s*){_move_pat}',
        text, re.IGNORECASE
    )
    if numbered:
        return [(s.upper(), d.upper()) for s, d in numbered]
    
    # === Strategy 3: Last compact move list on a single line ===
    # Look for lines containing 2+ comma/space-separated arrow moves
    for line in reversed(text.split('\n')):
        line = line.strip()
        found = re.findall(_move_pat, line, re.IGNORECASE)
        if len(found) >= 2:
            return [(s.upper(), d.upper()) for s, d in found]
    
    # === Strategy 4: Numbered "from X to Y" lines ===
    from_to = re.findall(
        r'(?:Move|Step)\s*\d+[:\.]?.*?from\s+(?:peg\s+)?([ABC])\s+to\s+(?:peg\s+)?([ABC])',
        text, re.IGNORECASE
    )
    if from_to:
        return [(s.upper(), d.upper()) for s, d in from_to]
    
    # === Strategy 5: Last MOVES: line with "from X to Y" ===
    if moves_match:
        line = moves_match.group(1)
        ft = re.findall(r'from\s+(?:peg\s+)?([ABC])\s+to\s+(?:peg\s+)?([ABC])', line, re.IGNORECASE)
        if ft:
            return [(s.upper(), d.upper()) for s, d in ft]
    
    return []


# ─── Move Validation ────────────────────────────────────────────────

def validate_solution(start_state, goal_state, moves) -> dict:
    """
    Validate a sequence of moves.
    Returns dict with: valid (bool), reached_goal (bool), n_moves, errors list.
    """
    state = deepcopy(start_state)
    errors = []

    for i, (src, dst) in enumerate(moves):
        if not state.get(src) or len(state[src]) == 0:
            errors.append(f"Move {i+1}: Peg {src} is empty")
            continue
        if len(state.get(dst, [])) >= PEG_CAPACITY.get(dst, 0):
            errors.append(f"Move {i+1}: Peg {dst} is full (capacity {PEG_CAPACITY[dst]})")
            continue
        ball = state[src].pop()
        state[dst].append(ball)

    reached_goal = state_to_tuple(state) == state_to_tuple(goal_state)

    return {
        "valid": len(errors) == 0,
        "reached_goal": reached_goal and len(errors) == 0,
        "n_moves": len(moves),
        "errors": errors,
        "final_state": state,
    }


# ─── Tier Configuration ────────────────────────────────────────────

TIERS = {
    "easy":   {"depths": [2],    "weight": 0.20},
    "medium": {"depths": [3],    "weight": 0.30},
    "hard":   {"depths": [4, 5], "weight": 0.50},
}


# ─── The Benchmark Task ────────────────────────────────────────────

@kbench.task(name="Tower of London")
def exec_func_tol(llm) -> float:
    """Tower of London Planning Benchmark.

    Tests multi-step planning by requiring the model to find move sequences
    to rearrange balls on pegs to match a goal state.

    Score = weighted sum of per-tier mean optimality:
    """
    results = []
    tier_scores = {"easy": [], "medium": [], "hard": []}

    for problem in TOL_PROBLEMS:
        start = problem["start"]
        goal = problem["goal"]
        optimal = problem["optimal_moves"]

        prompt = (
            f"TOWER OF LONDON PUZZLE — {problem['problem_id']}\n\n"
            f"Rules:\n"
            f"- 3 pegs (A, B, C) with capacity limits: A holds 3 balls, B holds 2, C holds 1\n"
            f"- Move only the TOP ball from one peg to another\n"
            f"- Goal: reach the goal state in as FEW moves as possible\n"
            f"- Optimal solution needs {optimal} moves\n\n"
            f"CURRENT STATE:\n{state_str(start)}\n\n"
            f"GOAL STATE:\n{state_str(goal)}\n\n"
            f"Think step by step. Plan your moves carefully.\n\n"
            f"CRITICAL: After your reasoning, you MUST end your response with exactly this format on its own line:\n"
            f"MOVES: A→B, C→A, B→C\n\n"
            f"Each move is SRC→DST (peg letter → peg letter). List all moves in order, separated by commas.\n"
            f"The MOVES: line must be the LAST line of your response."
        )

        with kbench.chats.new(f"tol_{problem['problem_id']}"):
            raw = llm.prompt(prompt)

        # Parse and validate
        moves = parse_moves(_strip_think(raw))
        validation = validate_solution(start, goal, moves)

        # Compute optimality ratio
        if validation["reached_goal"]:
            optimality = min(1.0, optimal / max(validation["n_moves"], 1))
        else:
            optimality = 0.0

        # Assign to tier
        if optimal in TIERS["easy"]["depths"]:
            tier_scores["easy"].append(optimality)
        elif optimal in TIERS["medium"]["depths"]:
            tier_scores["medium"].append(optimality)
        else:
            tier_scores["hard"].append(optimality)

        result = {
            "problem_id": problem["problem_id"],
            "optimal_moves": optimal,
            "model_moves": validation["n_moves"],
            "parsed_moves": len(moves),
            "reached_goal": validation["reached_goal"],
            "optimality": round(optimality, 4),
            "errors": validation["errors"],
        }
        results.append(result)

    # ── Compute Weighted Score ──
    tier_means = {}
    for tier_name, cfg in TIERS.items():
        scores = tier_scores[tier_name]
        tier_means[tier_name] = float(np.mean(scores)) if scores else 0.0

    score = sum(TIERS[t]["weight"] * tier_means[t] for t in TIERS)
    score = round(float(np.clip(score, 0, 1)), 4)

    # ── Log ──
    _safe_log({
        "benchmark": "Tower of London",
        "n_problems": len(results),
        "tier_means": {t: round(m, 4) for t, m in tier_means.items()},
        "tier_weights": {t: TIERS[t]["weight"] for t in TIERS},
        "composite_score": score,
        "per_problem": results,
    })

    return score

In [ ]:
exec_func_tol.run(llm=kbench.llm)